# Validación experimental sin z-score

Este cuaderno reproduce la comparación experimental usada para decidir la metodología de reconstrucción de la DSA, pero **sin normalizar mediante z-score**.

La comparación se realiza directamente en la escala de la DSA exportada por el BIS (`.f_a`) y de la DSA reconstruida, expresadas en dB. Por tanto:

- `MAE`, `RMSE` y `bias` están en dB;
- `Pearson` y `Spearman` se calculan sobre los valores reales en dB;
- no se usa `fun_dsa.zscore_global`;
- no se usa `fun_dsa.probar_suavizado_y_shifts`, porque esa función antigua normaliza internamente.

El registro ejecutado por defecto es `L03041035`, unilateral, porque contiene `.f_a`, `.spa`, `.h_a`, `.t_a` y onda cruda `.r2a`.


In [1]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.signal import welch, spectrogram, butter, sosfilt, sosfilt_zi
from scipy.stats import pearsonr, spearmanr

import pywt

from IPython.display import display, Markdown

NOTEBOOK_DIR = Path(r"C:\Users\usuario\TFG_BIS_GIS\notebooks")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

import funciones_aux as fau
import funciones_dsa as fun_dsa
import funciones_dsa_unilateral as fun_dsa_u
import funciones_dsa_bilateral as fun_dsa_b

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

print("Entorno listo.")


Entorno listo.


## 1. Parámetros de la prueba

In [2]:

RUTA_BASE = Path(r"C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035")

ruta_fa = RUTA_BASE / "L03041035.f_a"
ruta_spa = RUTA_BASE / "L03041035.spa"
ruta_ha = RUTA_BASE / "L03041035.h_a"
ruta_ta = RUTA_BASE / "L03041035.t_a"
ruta_r2a = RUTA_BASE / "L03041035.r2a"

FS = 128
VENTANA_WELCH_S = 2
PASO_S = 1
FMIN = 0.5
FMAX = 30.0
PASO_FREQ = 0.5
ANCHO_BIN_HZ = 0.5
REF_POTENCIA = 0.0001
EPS = 1e-12

VENTANAS_SUAVIZADO = [0, 1, 5, 10, 30, 60]
SHIFTS = range(0, 31)
USAR_BUTTERWORTH = [False, True]
METODOS = ["welch", "spectrogram", "wavelet"]
TIPOS_SUAVIZADO = ["sin_suavizado", "rolling", "ewm_alpha"]

UMBRAL_SQI = 15
UMBRAL_CEROS = 0.5

for ruta in [ruta_fa, ruta_spa, ruta_ha, ruta_ta, ruta_r2a]:
    print(("OK   " if ruta.exists() else "MISS ") + str(ruta))


OK   C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.f_a
OK   C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.spa
OK   C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.h_a
OK   C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.t_a
OK   C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.r2a


## 2. Funciones nuevas de comparación

Estas funciones son deliberadamente autocontenidas para evitar llamar a la función antigua `probar_suavizado_y_shifts`, que aplicaba z-score antes de calcular las métricas.


In [3]:

def filtrar_pasa_altos_causal(senal, fs=128, frecuencia_hz=0.25, orden=1):
    """
    Filtro Butterworth pasa-altos causal.
    Respeta bloques separados por NaN y no filtra a través de huecos.
    """
    senal = np.asarray(senal, dtype=float)
    if frecuencia_hz is None or frecuencia_hz <= 0:
        return senal.copy()

    salida = np.full_like(senal, np.nan, dtype=float)
    sos = butter(int(orden), float(frecuencia_hz), btype="highpass", fs=float(fs), output="sos")
    indices_validos = np.flatnonzero(np.isfinite(senal))
    if indices_validos.size == 0:
        return salida

    cortes = np.flatnonzero(np.diff(indices_validos) > 1) + 1
    bloques = np.split(indices_validos, cortes)
    for bloque in bloques:
        segmento = senal[bloque]
        zi = sosfilt_zi(sos) * segmento[0]
        filtrado, _ = sosfilt(sos, segmento, zi=zi)
        salida[bloque] = filtrado
    return salida


def convertir_potencia_bin_a_db(potencia_bin):
    return 10 * np.log10((np.asarray(potencia_bin, dtype=float) + EPS) / (REF_POTENCIA ** 2))


def frecuencias_objetivo():
    return np.arange(FMIN, FMAX + PASO_FREQ, PASO_FREQ)


def reconstruir_welch_db(senal, fs=FS):
    x = np.asarray(senal, dtype=float)
    nperseg = int(VENTANA_WELCH_S * fs)
    paso = int(PASO_S * fs)
    nfft = int(fs / PASO_FREQ)
    tiempos = []
    espectros = []

    for inicio in range(0, len(x) - nperseg + 1, paso):
        fin = inicio + nperseg
        segmento = x[inicio:fin]
        tiempo_s = (inicio + nperseg / 2) / fs
        tiempos.append(tiempo_s)

        if not np.isfinite(segmento).all():
            espectros.append(np.full(len(frecuencias_objetivo()), np.nan))
            continue

        f, pxx = welch(
            segmento,
            fs=fs,
            window="hann",
            nperseg=nperseg,
            noverlap=0,
            nfft=nfft,
            detrend="constant",
            scaling="density",
        )
        mask = (f >= FMIN) & (f <= FMAX)
        potencia_bin = pxx[mask] * ANCHO_BIN_HZ
        espectros.append(convertir_potencia_bin_a_db(potencia_bin))

    cols = [f"{f:.1f}" for f in frecuencias_objetivo()]
    df = pd.DataFrame(espectros, columns=cols)
    df.insert(0, "tiempo_s", tiempos)
    return df, frecuencias_objetivo()


def reconstruir_spectrogram_db(senal, fs=FS):
    x = np.asarray(senal, dtype=float)
    nperseg = int(VENTANA_WELCH_S * fs)
    noverlap = nperseg - int(PASO_S * fs)
    nfft = int(fs / PASO_FREQ)

    f, t, sxx = spectrogram(
        x,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        detrend="constant",
        scaling="density",
        mode="psd",
    )
    mask = (f >= FMIN) & (f <= FMAX)
    potencia_bin = sxx[mask, :].T * ANCHO_BIN_HZ
    dsa_db = convertir_potencia_bin_a_db(potencia_bin)
    cols = [f"{freq:.1f}" for freq in f[mask]]
    df = pd.DataFrame(dsa_db, columns=cols)
    df.insert(0, "tiempo_s", t)
    return df, f[mask]


def reconstruir_wavelet_db(senal, fs=FS):
    """
    Aproximación exploratoria por CWT.

    Nota metodológica:
    La potencia wavelet no es exactamente la PSD calibrada de Welch/spectrogram.
    Se fuerza a los mismos centros de frecuencia y a la misma escala de dB para
    poder estudiar si reproduce o no el archivo .f_a.
    """
    x = np.asarray(senal, dtype=float)
    freqs = frecuencias_objetivo()
    wavelet = "cmor1.5-1.0"
    central_freq = pywt.central_frequency(wavelet)
    scales = central_freq * fs / freqs
    coef, _ = pywt.cwt(x, scales, wavelet, sampling_period=1 / fs)
    potencia_inst = np.abs(coef) ** 2

    nperseg = int(VENTANA_WELCH_S * fs)
    paso = int(PASO_S * fs)
    tiempos = []
    espectros = []
    for inicio in range(0, len(x) - nperseg + 1, paso):
        fin = inicio + nperseg
        tiempo_s = (inicio + nperseg / 2) / fs
        tiempos.append(tiempo_s)
        ventana = potencia_inst[:, inicio:fin]
        potencia_bin = np.nanmean(ventana, axis=1) * ANCHO_BIN_HZ
        espectros.append(convertir_potencia_bin_a_db(potencia_bin))

    cols = [f"{f:.1f}" for f in freqs]
    df = pd.DataFrame(espectros, columns=cols)
    df.insert(0, "tiempo_s", tiempos)
    return df, freqs


def suavizar_dsa(dsa, tipo, ventana_s):
    if tipo == "sin_suavizado" or ventana_s == 0:
        return dsa.copy()
    if tipo == "rolling":
        return dsa.rolling(window=int(ventana_s), min_periods=1, center=False).mean()
    if tipo == "ewm_alpha":
        alpha = 1 / float(ventana_s)
        return dsa.ewm(alpha=alpha, adjust=False, ignore_na=True).mean()
    raise ValueError(f"Tipo de suavizado no reconocido: {tipo}")


def aplicar_shift(A, B, shift):
    if shift < 0:
        A2 = A.iloc[-shift:].reset_index(drop=True)
        B2 = B.iloc[:len(A2)].reset_index(drop=True)
    elif shift > 0:
        A2 = A.iloc[:-shift].reset_index(drop=True)
        B2 = B.iloc[shift:].reset_index(drop=True)
    else:
        A2 = A.reset_index(drop=True)
        B2 = B.reset_index(drop=True)

    n = min(len(A2), len(B2))
    return A2.iloc[:n], B2.iloc[:n]


def calcular_metricas_db(dsa_rec, dsa_fa):
    A = dsa_rec.to_numpy(dtype=float)
    B = dsa_fa.to_numpy(dtype=float)
    mask = np.isfinite(A) & np.isfinite(B)

    A_valid = A[mask]
    B_valid = B[mask]
    if len(A_valid) < 3:
        return {
            "n_celdas": len(A_valid),
            "Pearson": np.nan,
            "Spearman": np.nan,
            "MAE_dB": np.nan,
            "RMSE_dB": np.nan,
            "bias_rec_menos_fa_dB": np.nan,
        }

    diff = A_valid - B_valid
    return {
        "n_celdas": len(A_valid),
        "Pearson": pearsonr(A_valid, B_valid)[0],
        "Spearman": spearmanr(A_valid, B_valid)[0],
        "MAE_dB": np.mean(np.abs(diff)),
        "RMSE_dB": np.sqrt(np.mean(diff ** 2)),
        "bias_rec_menos_fa_dB": np.mean(diff),
    }


def probar_suavizado_y_shifts_sin_zscore(dsa_rec, dsa_fa, metodo, butterworth, tipo_suavizado):
    resultados = []
    for ventana in VENTANAS_SUAVIZADO:
        if tipo_suavizado != "sin_suavizado" and ventana == 0:
            continue
        if tipo_suavizado == "sin_suavizado" and ventana != 0:
            continue

        rec_suav = suavizar_dsa(dsa_rec, tipo_suavizado, ventana)
        for shift in SHIFTS:
            A, B = aplicar_shift(rec_suav, dsa_fa, shift)
            met = calcular_metricas_db(A, B)
            resultados.append({
                "metodo": metodo,
                "butterworth": butterworth,
                "suavizado": tipo_suavizado,
                "ventana_s": ventana,
                "shift_s": shift,
                **met,
            })
    return pd.DataFrame(resultados)


## 3. Lectura, alineación y máscara común

In [4]:

# DSA original .f_a
tiempo_fa, dsa_fa = fau.cargar_fa_directo(str(ruta_fa), escalar_db=True)

# Variables procesadas .spa
df_spa_raw = fau.procesar_spa(str(ruta_spa))
df_spa = fun_dsa_u.limpiar_spa_unilateral(df_spa_raw)
timeline_spa = fun_dsa_b.preparar_timeline_spa(df_spa, verbose=False)

# Ajuste de .f_a a timeline .spa
dsa_fa_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_fa,
    dsa=dsa_fa,
    timeline_spa=timeline_spa,
    nombre="DSA .f_a",
    verbose=False,
)

# Merge del .spa para máscara
df_merge = fun_dsa.alinear_spa_con_tiempo(timeline_spa, df_spa, resolver_duplicados="last")
_, mask_comun = fun_dsa.preparar_dsa_con_mask(
    tiempo=timeline_spa,
    dsa=dsa_fa_spa,
    df_merge=df_merge,
    umbral_sqi=UMBRAL_SQI,
    umbral_ceros=UMBRAL_CEROS,
    incluir_filas_nan=True,
)

dsa_fa_eval = dsa_fa_spa.copy()
dsa_fa_eval.loc[mask_comun.values, :] = np.nan

# EEG crudo
num_canales, fs, pendiente, offset = fau.extraer_parametros_eeg(str(ruta_ha))
df_eeg = fun_dsa_u.leer_r2a(str(ruta_r2a), pendiente, offset, fs=fs)
df_eeg_recortado, timeline_spa, info_alineacion = fun_dsa_b.recortar_raw_segun_ta_y_spa(
    df_raw=df_eeg,
    ruta_ta=str(ruta_ta),
    df_spa=df_spa,
    columna_time="Time",
    fs=fs,
    resolver_duplicados="last",
    verbose=False,
)

codigo_lofilter = int(pd.to_numeric(df_spa["LoFilter"], errors="coerce").dropna().mode().iloc[0])
MAPA_LOFILTER_HZ = {0: 0.25, 1: 1.0, 2: 2.0, 3: 2.5}
frecuencia_lofilter = MAPA_LOFILTER_HZ.get(codigo_lofilter, 0.25)

print("Registro:", RUTA_BASE.name)
print("fs:", fs)
print("LoFilter:", codigo_lofilter, "->", frecuencia_lofilter, "Hz")
print("Timeline .spa:", len(timeline_spa), timeline_spa.iloc[0], "->", timeline_spa.iloc[-1])
print("Filas enmascaradas:", int(mask_comun.sum()), "/", len(mask_comun))


Registro: DH03041035
fs: 128
LoFilter: 3 -> 2.5 Hz
Timeline .spa: 2119 2026-03-04 10:35:21 -> 2026-03-04 11:10:39
Filas enmascaradas: 146 / 2119


## 4. Reconstrucción por método y filtro

In [5]:

def reconstruir_metodo(metodo, senal):
    if metodo == "welch":
        return reconstruir_welch_db(senal, fs=fs)
    if metodo == "spectrogram":
        return reconstruir_spectrogram_db(senal, fs=fs)
    if metodo == "wavelet":
        return reconstruir_wavelet_db(senal, fs=fs)
    raise ValueError(metodo)


def adaptar_y_enmascarar(df_dsa, frecuencias, nombre):
    tiempo_tmp, dsa_tmp = fun_dsa.adaptar_dsa_reconstruida_para_plot(
        df_dsa=df_dsa,
        frecuencias=frecuencias,
        hora_inicio=timeline_spa.iloc[0],
        insertar_fila_inicial_nan=True,
    )
    dsa_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
        tiempo_dsa=tiempo_tmp,
        dsa=dsa_tmp,
        timeline_spa=timeline_spa,
        nombre=nombre,
        verbose=False,
    )
    dsa_eval = dsa_spa.copy()
    dsa_eval.loc[mask_comun.values, :] = np.nan
    return dsa_eval


reconstrucciones = {}
senal_base = df_eeg_recortado["canal_1_uV"].to_numpy(dtype=float)

for usar_butter in USAR_BUTTERWORTH:
    if usar_butter:
        senal = filtrar_pasa_altos_causal(
            senal_base,
            fs=fs,
            frecuencia_hz=frecuencia_lofilter,
            orden=1,
        )
    else:
        senal = senal_base.copy()

    for metodo in METODOS:
        print(f"Reconstruyendo {metodo}, Butterworth={usar_butter}...")
        df_dsa_metodo, freqs = reconstruir_metodo(metodo, senal)
        dsa_eval = adaptar_y_enmascarar(
            df_dsa_metodo,
            freqs,
            nombre=f"{metodo} Butterworth={usar_butter}",
        )
        reconstrucciones[(metodo, usar_butter)] = dsa_eval

print("Reconstrucciones listas:", len(reconstrucciones))


Reconstruyendo welch, Butterworth=False...


Reconstruyendo spectrogram, Butterworth=False...
Reconstruyendo wavelet, Butterworth=False...


Reconstruyendo welch, Butterworth=True...


Reconstruyendo spectrogram, Butterworth=True...


Reconstruyendo wavelet, Butterworth=True...


Reconstrucciones listas: 6


## 5. Búsqueda de suavizado y shift sin z-score

In [6]:

resultados = []
for (metodo, usar_butter), dsa_rec in reconstrucciones.items():
    for tipo_suavizado in TIPOS_SUAVIZADO:
        df_resultado = probar_suavizado_y_shifts_sin_zscore(
            dsa_rec=dsa_rec,
            dsa_fa=dsa_fa_eval,
            metodo=metodo,
            butterworth=usar_butter,
            tipo_suavizado=tipo_suavizado,
        )
        resultados.append(df_resultado)

df_resultados = pd.concat(resultados, ignore_index=True)

OUT_CSV = NOTEBOOK_DIR / "validacion_metodologias_sin_zscore_resultados.csv"
df_resultados.to_csv(OUT_CSV, index=False)

print("Filas evaluadas:", len(df_resultados))
print("CSV guardado en:", OUT_CSV)
display(df_resultados.head())


Filas evaluadas: 2046
CSV guardado en: C:\Users\usuario\TFG_BIS_GIS\notebooks\validacion_metodologias_sin_zscore_resultados.csv


,metodo,butterworth,suavizado,ventana_s,shift_s,n_celdas,Pearson,Spearman,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,welch,False,sin_suavizado,0,0,118380,0.409215,0.333040,7.890390,10.287474,3.919256
1,welch,False,sin_suavizado,0,1,117600,0.413302,0.336596,7.861484,10.242709,3.893845
2,welch,False,sin_suavizado,0,2,117000,0.415987,0.339679,7.840565,10.222622,3.877291
3,welch,False,sin_suavizado,0,3,116580,0.417794,0.342019,7.820485,10.205599,3.860597
4,welch,False,sin_suavizado,0,4,116280,0.422049,0.346467,7.784939,10.164375,3.838765


## 6. Tablas resumen

In [7]:

columnas = [
    "metodo", "butterworth", "suavizado", "ventana_s", "shift_s",
    "n_celdas", "Pearson", "Spearman", "MAE_dB", "RMSE_dB", "bias_rec_menos_fa_dB"
]

df_top_rmse = (
    df_resultados[columnas]
    .sort_values(["RMSE_dB", "MAE_dB", "Pearson"], ascending=[True, True, False])
    .head(20)
    .reset_index(drop=True)
)

display(Markdown("### Top 20 global ordenado por RMSE en dB"))
display(df_top_rmse)

idx = df_resultados.groupby(["metodo", "butterworth"])["RMSE_dB"].idxmin()
df_mejor_por_metodo_filtro = (
    df_resultados.loc[idx, columnas]
    .sort_values(["RMSE_dB", "MAE_dB"])
    .reset_index(drop=True)
)

display(Markdown("### Mejor combinación por método y uso de Butterworth"))
display(df_mejor_por_metodo_filtro)

idx2 = df_resultados.groupby(["metodo", "butterworth", "suavizado"])["RMSE_dB"].idxmin()
df_mejor_por_suavizado = (
    df_resultados.loc[idx2, columnas]
    .sort_values(["metodo", "butterworth", "RMSE_dB"])
    .reset_index(drop=True)
)

display(Markdown("### Mejor combinación por método, filtro y tipo de suavizado"))
display(df_mejor_por_suavizado)


### Top 20 global ordenado por RMSE en dB

,metodo,butterworth,suavizado,ventana_s,shift_s,n_celdas,Pearson,Spearman,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,welch,True,rolling,30,11,117060,0.725813,0.679485,3.717672,5.020815,2.949663
1,spectrogram,True,rolling,30,11,117060,0.725813,0.679485,3.717672,5.020815,2.949663
2,welch,True,rolling,30,10,117180,0.726020,0.680569,3.711267,5.021854,2.951632
3,spectrogram,True,rolling,30,10,117180,0.726020,0.680569,3.711267,5.021854,2.951632
4,welch,True,rolling,30,12,116940,0.724683,0.677294,3.727566,5.024571,2.946646
5,spectrogram,True,rolling,30,12,116940,0.724683,0.677294,3.727566,5.024571,2.946646
6,welch,True,rolling,30,9,117300,0.725309,0.680533,3.708754,5.028256,2.953438
7,spectrogram,True,rolling,30,9,117300,0.725309,0.680533,3.708754,5.028256,2.953438
8,welch,True,rolling,30,13,116820,0.722591,0.673994,3.741505,5.034015,2.943851
9,spectrogram,True,rolling,30,13,116820,0.722591,0.673994,3.741505,5.034015,2.943851


### Mejor combinación por método y uso de Butterworth

,metodo,butterworth,suavizado,ventana_s,shift_s,n_celdas,Pearson,Spearman,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,spectrogram,True,rolling,30,11,117060,0.725813,0.679485,3.717672,5.020815,2.949663
1,welch,True,rolling,30,11,117060,0.725813,0.679485,3.717672,5.020815,2.949663
2,spectrogram,False,rolling,30,11,117060,0.704109,0.671736,4.526958,6.447736,3.815285
3,welch,False,rolling,30,11,117060,0.704109,0.671736,4.526958,6.447736,3.815285
4,wavelet,True,rolling,30,13,116820,0.690021,0.641736,18.753610,19.256752,18.753610
5,wavelet,False,rolling,30,13,116820,0.672145,0.633374,19.627463,20.389205,19.627463


### Mejor combinación por método, filtro y tipo de suavizado

,metodo,butterworth,suavizado,ventana_s,shift_s,n_celdas,Pearson,Spearman,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,spectrogram,False,rolling,30,11,117060,0.704109,0.671736,4.526958,6.447736,3.815285
1,spectrogram,False,ewm_alpha,30,10,117780,0.676316,0.648038,4.599347,6.541322,3.862374
2,spectrogram,False,sin_suavizado,0,24,112020,0.520829,0.451314,7.217170,9.501302,3.871108
3,spectrogram,True,rolling,30,11,117060,0.725813,0.679485,3.717672,5.020815,2.949663
4,spectrogram,True,ewm_alpha,30,11,117720,0.694010,0.654296,3.812652,5.160606,3.000672
5,spectrogram,True,sin_suavizado,0,24,112020,0.470281,0.424494,6.643006,8.584422,3.003916
6,wavelet,False,rolling,30,13,116820,0.672145,0.633374,19.627463,20.389205,19.627463
7,wavelet,False,ewm_alpha,60,5,118080,0.631840,0.588636,19.648630,20.438314,19.648630
8,wavelet,False,sin_suavizado,0,20,112920,0.603730,0.540329,19.691163,20.822200,19.648264
9,wavelet,True,rolling,30,13,116820,0.690021,0.641736,18.753610,19.256752,18.753610


## 7. Lectura rápida del resultado

In [8]:

mejor = df_top_rmse.iloc[0]

display(Markdown(
    f'''
**Mejor combinación global sin z-score**

- Método: `{mejor['metodo']}`
- Butterworth: `{mejor['butterworth']}`
- Suavizado: `{mejor['suavizado']}`
- Ventana: `{mejor['ventana_s']}` s
- Shift: `{mejor['shift_s']}` s
- Pearson: `{mejor['Pearson']:.4f}`
- MAE: `{mejor['MAE_dB']:.4f}` dB
- RMSE: `{mejor['RMSE_dB']:.4f}` dB
- Bias reconstruida - .f_a: `{mejor['bias_rec_menos_fa_dB']:.4f}` dB
'''
))

display(Markdown(
    '''
**Nota para la memoria:** esta tabla no compara patrones normalizados, sino diferencias directas en la escala de visualización de la DSA. Por eso MAE y RMSE son interpretables como error en dB.
'''
))



**Mejor combinación global sin z-score**

- Método: `welch`
- Butterworth: `True`
- Suavizado: `rolling`
- Ventana: `30` s
- Shift: `11` s
- Pearson: `0.7258`
- MAE: `3.7177` dB
- RMSE: `5.0208` dB
- Bias reconstruida - .f_a: `2.9497` dB



**Nota para la memoria:** esta tabla no compara patrones normalizados, sino diferencias directas en la escala de visualización de la DSA. Por eso MAE y RMSE son interpretables como error en dB.
